In [ ]:
%additional_python_modules matplotlib
%matplotlib inline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.close('all')
from matplotlib.ticker import FuncFormatter


CINZA, DESTAQUE, INK = '#B8C0C8', '#0F6E78', '#16222E'
plt.rcParams.update({'font.size': 11, 'axes.edgecolor': '#D6DCE0',
                     'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 110})

def reais(v, _=None):
    return f'R$ {v:,.0f}'.replace(',', '.')

def label_nivel(nivel, ordem):
    return 'Sênior/Staff' if (pd.notna(ordem) and ordem >= 4) else nivel

t1 = spark.read.table('dados_gold.agg_estrutura_distribuicao').toPandas()
t2 = spark.read.table('dados_gold.agg_salario_cargo').toPandas()
t3 = spark.read.table('dados_gold.agg_salario_senioridade').toPandas()
t4 = spark.read.table('dados_gold.agg_resumo_evolucao_ano').toPandas()

# --- Salvar graficos no S3 (artefatos/frente_1/graficos/) ---
import boto3, os
BUCKET = 'tech-challenge-014478672967'
PREFIXO_GRAFICOS = 'artefatos/frente_1/graficos'
os.makedirs('/tmp/graficos', exist_ok=True)
_s3 = boto3.client('s3')

def salvar(fig, nome):
    local = f'/tmp/graficos/{nome}'
    fig.savefig(local, bbox_inches='tight', dpi=150)
    _s3.upload_file(local, BUCKET, f'{PREFIXO_GRAFICOS}/{nome}')
    print(f'Salvo em s3://{BUCKET}/{PREFIXO_GRAFICOS}/{nome}')


In [ ]:
# GRAFICO 1 - Salario mediano por senioridade (evolucao por ano)
t3['nivel_label'] = t3.apply(lambda r: label_nivel(r['nivel_comparavel'], r['nivel_ordem']), axis=1)
piv = (t3.groupby(['nivel_ordem', 'nivel_label', 'ano_pesquisa'])['salario_mediano']
         .mean().reset_index()
         .pivot(index=['nivel_ordem', 'nivel_label'], columns='ano_pesquisa', values='salario_mediano')
         .sort_index())
labels = [l for _, l in piv.index]
fig, ax = plt.subplots(figsize=(9, 5))
anos = list(piv.columns); x = range(len(labels)); larg = 0.8 / len(anos)
for i, ano in enumerate(anos):
    cor = DESTAQUE if ano == max(anos) else CINZA
    offs = [xi + i * larg for xi in x]
    barras = ax.bar(offs, piv[ano].values, larg, label=str(ano), color=cor)
    if ano == max(anos):
        for b in barras:
            if pd.notna(b.get_height()):
                ax.text(b.get_x()+b.get_width()/2, b.get_height()+300,
                        reais(b.get_height()), ha='center', fontsize=9, color=INK)
ax.set_xticks([xi + larg*(len(anos)-1)/2 for xi in x]); ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(FuncFormatter(reais))
ax.set_title('O salário do sênior saltou; júnior e pleno ficaram estáveis',
             fontweight='bold', color=INK, loc='left')
ax.legend(title='Ano', frameon=False); plt.tight_layout(); salvar(fig, "01_salario_por_senioridade.png"); plt.show()

In [ ]:
# GRAFICO 2 - Top cargos por salario mediano (2025, amostra >= 30)
c = t2[(t2['ano_pesquisa'] == 2025) & (t2['quantidade_profissionais'] >= 30)].sort_values('salario_mediano').tail(10)
fig, ax = plt.subplots(figsize=(9, 5.5))
cores = [DESTAQUE if v == c['salario_mediano'].max() else CINZA for v in c['salario_mediano']]
ax.barh(c['cargo_atual'], c['salario_mediano'], color=cores)
for y, v in enumerate(c['salario_mediano']):
    ax.text(v+200, y, reais(v), va='center', fontsize=9, color=INK)
ax.xaxis.set_major_formatter(FuncFormatter(reais))
ax.set_title('Engenharia de ML lidera a remuneração no mercado de dados (2025)',
             fontweight='bold', color=INK, loc='left')
plt.tight_layout(); salvar(fig, "02_salario_por_cargo_2025.png"); plt.show()

In [ ]:
# GRAFICO 3 - Evolucao da composicao de senioridade (%)
sen = t1[t1['tipo_dimensao'] == 'Senioridade'].copy()
comp = sen.groupby(['ano_pesquisa', 'categoria'])['quantidade'].sum().reset_index()
tot = comp.groupby('ano_pesquisa')['quantidade'].transform('sum')
comp['pct'] = 100 * comp['quantidade'] / tot
piv2 = comp.pivot(index='ano_pesquisa', columns='categoria', values='pct')
ordem_niveis = [n for n in ['Júnior', 'Pleno', 'Sênior'] if n in piv2.columns]
piv2 = piv2[ordem_niveis]
fig, ax = plt.subplots(figsize=(8, 5))
cores3 = {'Júnior': CINZA, 'Pleno': '#7FB0B6', 'Sênior': DESTAQUE}
for nivel in ordem_niveis:
    ax.plot(piv2.index, piv2[nivel], marker='o', linewidth=2.5, color=cores3.get(nivel, CINZA), label=nivel)
    ax.text(piv2.index[-1]+0.05, piv2[nivel].iloc[-1], f'{piv2[nivel].iloc[-1]:.0f}%',
            va='center', fontsize=10, color=cores3.get(nivel, INK))
ax.set_xticks(piv2.index); ax.set_ylabel('% dos profissionais')
ax.set_title('O mercado amadurece: menos júniores, mais sêniores',
             fontweight='bold', color=INK, loc='left')
ax.legend(frameon=False); plt.tight_layout(); salvar(fig, "03_composicao_senioridade.png"); plt.show()

In [ ]:
# GRAFICO 4 - Distribuicao por regiao (2025)
reg = t1[(t1['tipo_dimensao'] == 'Regiao') & (t1['ano_pesquisa'] == 2025)].sort_values('quantidade')
fig, ax = plt.subplots(figsize=(8, 4.5))
cores4 = [DESTAQUE if v == reg['quantidade'].max() else CINZA for v in reg['quantidade']]
ax.barh(reg['categoria'], reg['pct_no_ano'], color=cores4)
for y, v in enumerate(reg['pct_no_ano']):
    ax.text(v+0.5, y, f'{v:.1f}%', va='center', fontsize=9, color=INK)
ax.set_xlabel('% dos profissionais')
ax.set_title('O mercado de dados é fortemente concentrado no Sudeste (2025)',
             fontweight='bold', color=INK, loc='left')
plt.tight_layout(); salvar(fig, "04_distribuicao_regiao_2025.png"); plt.show()

In [ ]:
# GRAFICO 5 - Evolucao da idade media por ano (usa agg_resumo_evolucao_ano)
t4s = t4.sort_values('ano_pesquisa')
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(t4s['ano_pesquisa'], t4s['idade_media'], marker='o', linewidth=2.5, color=DESTAQUE)
for _, r in t4s.iterrows():
    ax.text(r['ano_pesquisa'], r['idade_media']+0.05, f"{r['idade_media']:.1f}", ha='center', fontsize=10, color=INK)
ax.set_xticks(t4s['ano_pesquisa']); ax.set_ylabel('Idade média')
ax.set_title('O profissional de dados está ficando um pouco mais velho a cada ano',
             fontweight='bold', color=INK, loc='left')
plt.tight_layout(); salvar(fig, "05_idade_media_por_ano.png"); plt.show()